# Use cases

Three concrete lookups, run against real data:

1. **per compound** — all targets it has been tested against
2. **per target** — all compounds tested against it
3. **per protein family** — which compounds are available for it

`probe.db` is built from `staging/moa`. Rebuild it with:

```bash
uv run python examples/populate_db.py
```

Delete any existing `probe.db` first -- `ProbeDB(..., create=True)` refuses to
run against a database that already has tables.

Real ChEMBL data is messy: units and endpoint types vary widely (IC50, Ki,
AC50, Potency, LogP, Cmax, ...), and some "targets" are actually assay
organisms or metadata placeholders (`Homo sapiens`, `ADMET`, `Unchecked`).
The code below picks a potency scale dynamically per query and shows target
names as-is rather than cleaning them up.

In [1]:
from pathlib import Path

import pandas as pd

from probedb import ProbeDB

pd.set_option("display.max_colwidth", 44)
pd.set_option("display.width", 170)

DB_PATH = Path("..") / "probe.db"
assert DB_PATH.exists(), f"{DB_PATH} not found -- run examples/populate_db.py first"

db = ProbeDB(DB_PATH, create=False)

db.counts()

,table,rows
0,compound,16573
1,chembl,15219
2,uniprot,6068
3,target,6908
4,target_uniprot,8584
5,bioactivity_source,51228
6,bioactivity_group,268911
7,bioactivity,457060


## Use case 1: per compound

For every compound with data, show:

- which targets it's been measured against, and which source(s) the data comes from
- its **main target** — strongest potency on a recognized scale (`IC50`, `EC50`,
  `AC50`, `Ki`, `Kd`, `GI50`, `CC50`, `ED50`, `Potency`; `relation == "="`, real unit)
- its **selectivity** — how much weaker the next best target is on that same scale

Bounded/censored measurements (`>`, `<`, ...) are shown separately, not pooled
into the ranking. Shows the `TOP_N` compounds with the most measurements on file.

In [2]:
POTENCY_TYPES = {"IC50", "EC50", "AC50", "Ki", "Kd", "GI50", "CC50", "ED50", "Potency"}
TOP_N = 10  # cap how many rows/entries get printed per section


def compound_label(db, inchikey):
    # this source leaves compound.name empty, so fall back to a ChEMBL id
    # and finally the InChIKey itself -- always something identifiable
    name = db.one("SELECT name FROM compound WHERE inchikey = ?", inchikey)
    chembl_id = db.one("SELECT chembl_id FROM chembl WHERE inchikey = ?", inchikey)
    return name or chembl_id or inchikey


def numeric_potency_rows(hits):
    return hits[
        hits.bioactivity_type.isin(POTENCY_TYPES)
        & (hits.relation == "=")
        & hits.value.notna()
        & (hits.unit != "unspecified")
    ]


def compound_profile(db, compound):
    hits = db.bioactivities(compound=compound)

    targets = hits[["target_type", "target"]].drop_duplicates().reset_index(drop=True)
    sources = sorted(hits["source_db"].dropna().unique())

    numeric = numeric_potency_rows(hits)
    scale, potency = None, pd.DataFrame(columns=["target", "target_type", "value"])
    if not numeric.empty:
        scale = numeric.groupby(["bioactivity_type", "unit"]).size().idxmax()
        comparable = numeric[
            (numeric.bioactivity_type == scale[0]) & (numeric.unit == scale[1])
        ]
        potency = (
            comparable.groupby(["target", "target_type"], as_index=False)["value"]
            .median()
            .sort_values("value")
            .reset_index(drop=True)
        )

    counter_screens = hits[hits.relation.isin([">", ">=", "<", "<="])]

    return targets, sources, scale, potency, counter_screens


profiled = sorted(db.table("bioactivity")["inchikey"].unique())
richness = db.table("bioactivity").groupby("inchikey").size().sort_values(ascending=False)
featured = list(richness.head(TOP_N).index)

print(f"{len(profiled)} of {len(db.table('compound'))} compounds have any bioactivity on file")
print(f"showing the {len(featured)} with the most measurements on file\n")

for inchikey in featured:
    label = compound_label(db, inchikey)
    targets, sources, scale, potency, counter_screens = compound_profile(db, inchikey)

    print(f"== {label} ==")
    print(f"targets measured: {len(targets)} (showing up to {TOP_N})")
    print(targets.head(TOP_N).to_string(index=False))

    print(f"derived from: {', '.join(sources) if sources else 'no source recorded'}")

    if potency.empty:
        print("main target: no recognized potency measurement "
              "(IC50/EC50/Ki/Kd/AC50/Potency/...) on a named scale")
    else:
        btype, unit = scale
        best = potency.iloc[0]
        print(f"main target ({btype}, {unit}, {len(potency)} targets on this scale): "
              f"{best.target}  ({best.value:g} {unit})")
        if len(potency) > 1:
            second = potency.iloc[1]
            fold = second.value / best.value
            print(
                f"selectivity: {fold:.1f}-fold vs {second.target} "
                f"({second.value:g} {unit}), the next best on the same scale"
            )
        else:
            print(f"selectivity: only one target with comparable {btype} ({unit}) data")

    if not counter_screens.empty:
        print(f"bounded/censored measurements: {len(counter_screens)} rows on other scales "
              f"(showing up to {TOP_N}), read qualitatively:")
        print(
            counter_screens[["target", "bioactivity_type", "relation", "value", "unit"]]
            .head(TOP_N)
            .to_string(index=False)
        )

    print()

12939 of 16573 compounds have any bioactivity on file
showing the 10 with the most measurements on file

== Acetazolamide ==
targets measured: 112 (showing up to 10)
target_type                target
    protein  Carbonic anhydrase 1
    protein Carbonic anhydrase 13
    protein  Carbonic anhydrase 2
    protein  Carbonic anhydrase 9
    protein                   CA1
    protein                   CA2
    protein                   CA9
    protein  Carbonic anhydrase 4
    protein                   CA4
     family    Carbonic anhydrase
derived from: ChEMBL
main target (Ki, nM, 34 targets on this scale): Carbonic anhydrase  (0.8 nM)
selectivity: 2.2-fold vs Carbonic anhydrase V (1.8 nM), the next best on the same scale
bounded/censored measurements: 67 rows on other scales (showing up to 10), read qualitatively:
                                                   target bioactivity_type relation   value unit
                                                      CA2             IC50        

## Use case 2: per target

The mirror image of use case 1:

- how many compounds tested, and which source(s) they come from
- **most potent** compound on a recognized scale
- **most selective** compound — the one for which this target is its best hit
  by the widest margin vs. its own next-best target (reuses `compound_profile`
  from use case 1)

Shows the coverage distribution, then the `TOP_N` targets with the most
measurements on file.

In [3]:
def target_profile(db, target_id):
    hits = db.bioactivities(target=target_id)

    compound_keys = sorted(hits["inchikey"].unique())
    sources = sorted(hits["source_db"].dropna().unique())

    numeric = numeric_potency_rows(hits)
    scale, potency = None, pd.DataFrame(columns=["inchikey", "value"])
    if not numeric.empty:
        scale = numeric.groupby(["bioactivity_type", "unit"]).size().idxmax()
        comparable = numeric[
            (numeric.bioactivity_type == scale[0]) & (numeric.unit == scale[1])
        ]
        potency = (
            comparable.groupby("inchikey", as_index=False)["value"]
            .median()
            .sort_values("value")
            .reset_index(drop=True)
        )

    return compound_keys, sources, scale, potency


def target_preference(db, compound, target_name):
    # how this target compares to `compound`'s own best *other* target,
    # reusing the per-compound potency ranking from use case 1
    _, _, _, potency, _ = compound_profile(db, compound)
    at_target = potency[potency.target == target_name]
    others = potency[potency.target != target_name]
    if at_target.empty or others.empty:
        return None
    return others.value.min() / at_target.value.iloc[0]


targeted = db.table("bioactivity").groupby("target_id")["inchikey"].nunique()
row_counts = db.table("bioactivity").groupby("target_id").size()

print(f"{len(targeted)} of {len(db.table('target'))} targets have at least one measurement")
print(targeted.value_counts().sort_index().rename_axis("compounds tested").rename("targets"))
print()

featured = list(row_counts.sort_values(ascending=False).head(TOP_N).index)

print(f"showing the {len(featured)} targets with the most measurements on file\n")

targets_tbl = db.table("target").set_index("target_id")

for target_id in featured:
    target = targets_tbl.loc[int(target_id)]
    compound_keys, sources, scale, potency = target_profile(db, int(target_id))

    print(f"== {target['name']} ({target.type}) ==")
    print(f"compounds tested ({len(compound_keys)}): "
          f"{', '.join(compound_label(db, k) for k in compound_keys[:TOP_N])}"
          f"{', ...' if len(compound_keys) > TOP_N else ''}")
    print(f"derived from: {', '.join(sources) if sources else 'no source recorded'}")

    if potency.empty:
        print("most potent: no recognized potency measurement for this target")
    else:
        btype, unit = scale
        best = potency.iloc[0]
        print(f"most potent ({btype}, {unit}): "
              f"{compound_label(db, best.inchikey)}  ({best.value:g} {unit})")

        ratios = [
            (compound_label(db, k), target_preference(db, k, target["name"]))
            for k in potency["inchikey"]
        ]
        ratios = [(c, r) for c, r in ratios if r is not None]
        if not ratios:
            print("most selective: no compound has another comparable target to compare against")
        else:
            top_compound, top_ratio = max(ratios, key=lambda cr: cr[1])
            if top_ratio > 1:
                print(f"most selective: {top_compound}  ({top_ratio:.1f}-fold vs its next best target)")
            else:
                print(
                    f"most selective: none, really -- even {top_compound} is "
                    f"{1 / top_ratio:.1f}-fold more potent on a different target"
                )

    print()

6304 of 6908 targets have at least one measurement
compounds tested
1       2210
2        813
3        450
4        329
5        247
        ... 
1785       1
1802       1
2040       1
3096       1
3115       1
Name: targets, Length: 411, dtype: int64

showing the 10 targets with the most measurements on file

== HDAC6 (protein) ==
compounds tested (3115): molibresib, DAG, sophocarpine, Tarceva, diazepam, butenafine hydrochloride, LY-3023414, bilastine, Ismelin, Hexachlorophene, ...
derived from: ChEMBL, Chemical Probes Portal, SPARK-PNAS
most potent (IC50, nM): Dactolisib  (2 nM)


most selective: Heparit  (51.5-fold vs its next best target)

== Replicase polyprotein 1ab (protein) ==
compounds tested (3096): molibresib, DAG, sophocarpine, Tarceva, diazepam, AF-220, LY-3023414, bilastine, Hexachlorophene, Ivabradine, ...
derived from: ChEMBL
most potent (IC50, nM): Xocova  (5.0065 nM)


most selective: Zinc Pyrithione  (240.0-fold vs its next best target)

== KCNH2 (protein) ==
compounds tested (2040): molibresib, Tarceva, diazepam, bretylium tosylate, butenafine hydrochloride, FR-257516, ibuprofen piconol, Ismelin, Hexachlorophene, Ivabradine, ...
derived from: ChEMBL, SPARK-PNAS
most potent (IC50, nM): BTRX-335140  (1 nM)


most selective: ibutilide fumarate  (14.5-fold vs its next best target)

== EGFR (protein) ==
compounds tested (476): Tarceva, Hexachlorophene, Balimek, AEW541, GSK-2110183, ELLAGIC ACID, Palbociclib, EMD 1214063, brigatinib, XL-228, ...
derived from: ChEMBL, Chemical Probes Portal, EUbOPEN, SPARK-PNAS
most potent (IC50, nM): IMATINIB  (0.11 nM)


most selective: FALNIDAMOL  (567.2-fold vs its next best target)

== TDP1 (protein) ==
compounds tested (1196): Tarceva, diazepam, butenafine hydrochloride, Hexachlorophene, PHTPP, ELLAGIC ACID, Lipoic acid, Quinethazone, orlistat, bentazepam, ...
derived from: ChEMBL, SPARK-PNAS
most potent (Potency, nM): Carbetapentane  (1 nM)


most selective: Carbetapentane  (316.2-fold vs its next best target)

== ADRA2A (protein) ==
compounds tested (1802): Tarceva, diazepam, bretylium tosylate, butenafine hydrochloride, ibuprofen piconol, Ismelin, Hexachlorophene, Ivabradine, Clodronic Acid, ubidecarenone, ...
derived from: ChEMBL, EUbOPEN, SPARK-PNAS
most potent (AC50, nM): brimonidine tartrate  (2.4 nM)


most selective: Trivastal Retard  (10.2-fold vs its next best target)

== ADRA1A (protein) ==
compounds tested (1703): Tarceva, diazepam, bretylium tosylate, butenafine hydrochloride, ibuprofen piconol, Ismelin, Hexachlorophene, Ivabradine, Clodronic Acid, ubidecarenone, ...
derived from: ChEMBL, EUbOPEN
most potent (AC50, nM): naphazoline hydrochloride  (1 nM)


most selective: methoxamine hydrochloride  (1235.1-fold vs its next best target)

== DRD1 (protein) ==
compounds tested (1785): molibresib, Tarceva, diazepam, bretylium tosylate, butenafine hydrochloride, ibuprofen piconol, Ismelin, Hexachlorophene, Ivabradine, Clodronic Acid, ...
derived from: ChEMBL, EUbOPEN, SPARK-PNAS
most potent (AC50, nM): Taractan  (3 nM)


most selective: clotiapine  (29.2-fold vs its next best target)

== ESR1 (protein) ==
compounds tested (1686): Tarceva, diazepam, bretylium tosylate, butenafine hydrochloride, ibuprofen piconol, Ismelin, Hexachlorophene, Ivabradine, Clodronic Acid, ubidecarenone, ...
derived from: ChEMBL, Chemical Probes Portal, SPARK-PNAS
most potent (IC50, nM): GIREDESTRANT  (0.24 nM)
most selective: GIREDESTRANT  (25416.7-fold vs its next best target)

== DRD2 (protein) ==
compounds tested (1245): molibresib, Tarceva, diazepam, bretylium tosylate, SEP-363856, butenafine hydrochloride, ubidecarenone, Balimek, BNAG, ETP, ...
derived from: ChEMBL, EUbOPEN, SPARK-PNAS
most potent (Ki, nM): benperidol  (0.027 nM)


most selective: sultopride  (118.2-fold vs its next best target)



## Use case 3: per protein family

Which compounds are available for a family target (`target.type == "family"`)?

`staging/moa` has no family targets -- everything loads as `protein`. This
falls back to `staging/_template`, the one place in the repo that defines a
family (`PARP 1, 2 and 3`), to show the same `target_profile` lookup running.
The schema has no `probe`/`chemogenomic`/`drug` column, so "available" means
what's on file: identifier, SMILES, and potency.

In [4]:
families = db.table("target")
families = families[families.type == "family"]
families

,target_id,type,name
2395,2396,family,Histone-lysine N-methyltransferase EHMT1...
2403,2404,family,CREB-binding protein/Histone acetyltrans...
2480,2481,family,Glycogen synthase kinase-3
2574,2575,family,Heat shock protein HSP90
2586,2587,family,Tankyrase 1/2
...,...,...,...
6795,6796,family,Voltage-gated potassium channel
6801,6802,family,Caspase
6804,6805,family,RAC1/RAC2/RAC3
6828,6829,family,Bombesin receptor


In [5]:
from loader import load
from loader.load import STAGING as STAGING_ROOT

if families.empty:
    print("No family-type targets in probe.db -- staging/moa loads everything as a "
          "single protein. Falling back to staging/_template to show the lookup working.\n")
    demo = ProbeDB(":memory:", create=True)
    load(demo, STAGING_ROOT / "_template", source="template")
    families = demo.table("target")
    families = families[families.type == "family"]
else:
    demo = db

FAMILY_NAME = families.iloc[0]["name"]  # swap in "RAS" or any other family name once one is loaded
family_id = int(families.iloc[0].target_id)

compound_keys, sources, scale, potency = target_profile(demo, family_id)

identity = demo.table("compound").merge(demo.table("chembl"), on="inchikey", how="left")
identity = identity[identity["inchikey"].isin(compound_keys)]

print(f"== {FAMILY_NAME} (family) ==")
print(f"compounds available ({len(compound_keys)}), derived from: {', '.join(sources)}")
print()

for _, row in identity.iterrows():
    chembl = row.chembl_id if pd.notna(row.chembl_id) else "no ChEMBL id on file"
    hit = potency[potency.inchikey == row.inchikey]
    if not hit.empty and scale:
        btype, unit = scale
        potency_str = f"{hit.value.iloc[0]:g} {unit} ({btype})"
    else:
        potency_str = "no comparable potency data"

    label = row["name"] or chembl
    print(label)
    print(f"  inchikey: {row.inchikey}")
    print(f"  chembl:   {chembl}")
    print(f"  smiles:   {row.smiles[:40]}...")
    print(f"  potency on {FAMILY_NAME}: {potency_str}")
    print()

== Histone-lysine N-methyltransferase EHMT1/EHMT2 (family) ==
compounds available (1), derived from: Probes & Drugs

UNC0638
  inchikey: QOECJCJVIMVJGX-UHFFFAOYSA-N
  chembl:   CHEMBL1231795
  smiles:   COc1cc2c(NC3CCN(C(C)C)CC3)nc(C3CCCCC3)nc...
  potency on Histone-lysine N-methyltransferase EHMT1/EHMT2: no comparable potency data

